# 煙霧偵測模型訓練 — Google Colab

**流程：** 下載資料集 → albumentations 擴增至 ~1800 張 → YOLOv8s 訓練 → 匯出 ONNX

**開始前：** 上方選單 → `執行階段 → 變更執行階段類型 → T4 GPU`

## 0. 確認 GPU

In [ ]:
!nvidia-smi

## 1. 安裝套件

In [ ]:
!pip install -q roboflow ultralytics onnx onnxsim albumentations

## 2. 下載主資料集

填入你的 Roboflow API Key：[取得方式](https://app.roboflow.com/) → Settings → Roboflow API

In [ ]:
from roboflow import Roboflow

API_KEY = "YOUR_ROBOFLOW_API_KEY"  # ← 替換為你的 Key

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("naruesuan-university").project("smoke-gxoy3")
versions = project.versions()
print(f"可用版本：{[v.version for v in versions]}")
latest = versions[-1]
dataset = latest.download("yolov8", location="/content/smoke_dataset")
print(f"✅ 下載完成：{dataset.location}")

## 3. 資料增強（438 張 → ~1800 張）

Roboflow API 僅能存取自己的 workspace，改用 albumentations 離線增強擴充資料量。  
針對煙霧特性加入 RandomFog、亮度/對比、模糊等增強，每張產生 3 個變體。

In [ ]:
import os, glob, cv2, random, shutil
import numpy as np
import albumentations as A

# 自動偵測來源目錄
BASE = "/content/smoke_dataset"
SRC_IMG, SRC_LBL = None, None
for img_cand, lbl_cand in [("train/images", "train/labels"), ("images/train", "labels/train")]:
    if glob.glob(f"{BASE}/{img_cand}/*.jpg") or glob.glob(f"{BASE}/{img_cand}/*.png"):
        SRC_IMG, SRC_LBL = f"{BASE}/{img_cand}", f"{BASE}/{lbl_cand}"
        break
assert SRC_IMG, "❌ 找不到影像目錄，請確認 Cell 2 已執行"
print(f"來源：{SRC_IMG}")

AUG_DIR = "/content/smoke_augmented"
AUG_PER_IMAGE = 3
for sub in ["images/train", "images/val", "labels/train", "labels/val"]:
    os.makedirs(f"{AUG_DIR}/{sub}", exist_ok=True)

augment = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=0.35, contrast_limit=0.35, p=0.8),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=40, val_shift_limit=30, p=0.7),
    A.GaussNoise(p=0.4),
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    A.RandomFog(fog_coef_lower=0.1, fog_coef_upper=0.4, p=0.35),
    A.CLAHE(p=0.3),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.2, rotate_limit=12, p=0.6),
], bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"], min_visibility=0.3))

def read_labels(path):
    boxes, classes = [], []
    if os.path.exists(path):
        for line in open(path):
            p = line.strip().split()
            if len(p) == 5:
                classes.append(int(p[0]))
                boxes.append([float(x) for x in p[1:]])
    return boxes, classes

def write_labels(path, boxes, classes):
    with open(path, "w") as f:
        for cls, box in zip(classes, boxes):
            f.write(f"{cls} {' '.join(f'{v:.6f}' for v in box)}\n")

imgs = sorted(glob.glob(f"{SRC_IMG}/*.jpg") + glob.glob(f"{SRC_IMG}/*.png"))
random.seed(42); random.shuffle(imgs)
val_split  = max(5, int(len(imgs) * 0.1))
val_imgs   = imgs[:val_split]
train_imgs = imgs[val_split:]

total = 0
for subset, img_list in [("train", train_imgs), ("val", val_imgs)]:
    for img_path in img_list:
        stem = os.path.splitext(os.path.basename(img_path))[0]
        lbl  = os.path.join(SRC_LBL, stem + ".txt")
        img  = cv2.imread(img_path)
        if img is None: continue
        shutil.copy(img_path, f"{AUG_DIR}/images/{subset}/{stem}.jpg")
        if os.path.exists(lbl):
            shutil.copy(lbl, f"{AUG_DIR}/labels/{subset}/{stem}.txt")
        total += 1
        if subset == "train":
            boxes, classes = read_labels(lbl)
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            for i in range(AUG_PER_IMAGE):
                try:
                    r = augment(image=img_rgb, bboxes=boxes, class_labels=classes)
                    cv2.imwrite(f"{AUG_DIR}/images/train/{stem}_aug{i}.jpg",
                                cv2.cvtColor(r["image"], cv2.COLOR_RGB2BGR))
                    write_labels(f"{AUG_DIR}/labels/train/{stem}_aug{i}.txt",
                                 r["bboxes"], r["class_labels"])
                    total += 1
                except Exception:
                    pass

n_tr = len(glob.glob(f"{AUG_DIR}/images/train/*.jpg"))
n_v  = len(glob.glob(f"{AUG_DIR}/images/val/*.jpg"))
print(f"✅ 增強完成：train={n_tr} 張，val={n_v} 張（原始 {len(imgs)} 張）")

## 4. 建立 data.yaml

In [ ]:
import yaml

AUG_DIR = "/content/smoke_augmented"
DATA_YAML = f"{AUG_DIR}/data.yaml"

with open(DATA_YAML, "w") as f:
    yaml.dump({
        "path":  AUG_DIR,
        "train": "images/train",
        "val":   "images/val",
        "nc":    1,
        "names": ["smoke"],
    }, f, default_flow_style=False)

print("✅ data.yaml：")
!cat /content/smoke_augmented/data.yaml

## 5. 訓練模型（YOLOv8s）

T4 GPU 大約需要 **30–50 分鐘**

In [ ]:
import torch, os
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else "cpu"
print(f"裝置：{'GPU (' + torch.cuda.get_device_name(0) + ')' if device == 0 else 'CPU'}")

DATA_YAML = "/content/smoke_augmented/data.yaml"
model = YOLO("yolov8s.pt")
model.train(
    data=DATA_YAML,
    epochs=100,
    batch=16 if device == 0 else 4,
    imgsz=640,
    device=device,
    project="/content/runs",
    name="smoke_detector",
    patience=20,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10.0, translate=0.1, scale=0.5,
    fliplr=0.5, mosaic=1.0, mixup=0.1, copy_paste=0.1,
    save=True, save_period=10, exist_ok=True,
)

## 6. 驗證指標

In [ ]:
import glob as _glob
from ultralytics import YOLO
from IPython.display import Image as IPImage, display

DATA_YAML = "/content/smoke_augmented/data.yaml"
candidates = sorted(_glob.glob("/content/runs/smoke_detector*/weights/best.pt"))
assert candidates, "❌ 找不到 best.pt，請確認訓練已完成"
best_pt = candidates[-1]
print(f"模型：{best_pt}")

metrics = YOLO(best_pt).val(data=DATA_YAML)
print(f"\nmAP50    : {metrics.box.map50:.4f}")
print(f"mAP50-95 : {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall   : {metrics.box.mr:.4f}")

for p in sorted(_glob.glob("/content/runs/smoke_detector*/*.png")):
    display(IPImage(p))

## 7. 匯出 ONNX

In [ ]:
from ultralytics import YOLO
YOLO(best_pt).export(format="onnx", imgsz=640, simplify=True, opset=17)
onnx_path = best_pt.replace(".pt", ".onnx")
print(f"✅ ONNX：{onnx_path}")

## 8. 下載模型

In [ ]:
import shutil, os
from google.colab import files

dst = "/content/smoke_detector.onnx"
shutil.copy(onnx_path, dst)
print(f"大小：{os.path.getsize(dst)/1024/1024:.1f} MB")
files.download(dst)
print("\n下載後複製到 cgr_detection/models/smoke_detector.onnx")

## 9. （選用）下載 .pt 權重備份

In [ ]:
from google.colab import files
files.download(best_pt)